# Huấn luyện mô hình MulCo End-to-End với Dữ liệu Tăng cường (Depth AUG)
Notebook này thực hiện Fine-tune mô hình MulCo trên tập dữ liệu `PlantDocSplited_depth_AUG`.
Tập dữ liệu này kết hợp ảnh gốc và ảnh đã xóa nền bằng Depth Anything V2, giúp mô hình mạnh mẽ hơn trước sự thay đổi của bối cảnh (Domain Shift).

**Các phép biến đổi dữ liệu (Augmentation):**
- Lật ngang và lật dọc ngẫu nhiên (p=0.5)
- Xoay ngẫu nhiên ±30°
- Thay đổi độ sáng và độ tương phản ±15%

In [24]:
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

# Setup Project Root
current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))
print(f"Project Root: {PROJECT_ROOT}")

# Thêm autoreload để Jupyter tự động cập nhật code khi file .py bên ngoài thay đổi
%load_ext autoreload
%autoreload 2

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset, multimodal_raw_collate_fn
from src.models.backbones.vision.convnext_cbam import ConvNeXt_CBAM
from src.models.fusion.mulco_fusion import MulCoFusionBlock
from src.models.multimodal.mulco_classifier import Conv1x1Classifier

Project Root: /media/data3/users/luongdth/MulCo-PlantNet
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Định nghĩa Data Augmentation & Load Dataset

In [25]:
import json
import re

# Tạo file mapping tự động cho CẢ TRAIN, VAL VÀ TEST
base_aug_dir = PROJECT_ROOT / "data/processed/PlantDocSplited_depth_AUG"
mapping_path = base_aug_dir / "global_image_caption_mapping.json"
mapping = {}

for split in ["train", "validation", "test"]:
    split_dir = base_aug_dir / split
    if split_dir.exists():
        for class_dir in split_dir.iterdir():
            if not class_dir.is_dir():
                continue
            class_name = class_dir.name
            for img_path in class_dir.glob("*.*"):
                if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                    continue
                img_name = img_path.name
                match = re.search(r'_(\d+)(_depth_suppressed)?\.(jpg|jpeg|png)$', img_name, re.IGNORECASE)
                if match:
                    num = int(match.group(1))
                    orig_name = f"{class_name}_{num:05d}.jpg"
                    mapping[f"{class_name}/{img_name}"] = orig_name

with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(mapping, f, indent=2)
print(f"Đã tạo mapping cho {len(mapping)} ảnh tại: {mapping_path}\n")

# Định nghĩa các phép biến đổi ảnh trong quá trình huấn luyện
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    # Thay RandomRotation bằng RandomAffine để kết hợp xoay, dịch chuyển và thu phóng an toàn
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15), # Thay đổi độ sáng/tương phản 15%
    # Thêm Gaussian Blur với xác suất thấp (20%) để học ảnh mờ mà không phá hỏng chi tiết bệnh nhỏ
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Tập Validation không dùng Augmentation, chỉ Resize và Normalize
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Giảm Batch Size từ 16 xuống 8 hoặc 4 để tránh lỗi CUDA Out of Memory khi dùng RoBERTa (256 tokens)
BATCH_SIZE = 8

print("Loading Training Dataset...")
train_dataset = MultiModalRawDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/processed/PlantDocSplited_depth_AUG/train"),
    caption_root=os.path.join(PROJECT_ROOT, "data/AIDG/captions_LLaVA"),
    transform=train_transform,
    strict_caption_match=False,
    image_caption_mapping_path=str(mapping_path)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=4, collate_fn=multimodal_raw_collate_fn,
                          drop_last=True)

print("\nLoading Validation Dataset...")
# Sử dụng tập validation chuẩn
val_dataset = MultiModalRawDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/processed/PlantDocSplited_depth_AUG/validation"),
    caption_root=os.path.join(PROJECT_ROOT, "data/AIDG/captions_LLaVA"),
    transform=val_transform,
    strict_caption_match=False,
    image_caption_mapping_path=str(mapping_path)
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=4, collate_fn=multimodal_raw_collate_fn)


Đã tạo mapping cho 5256 ảnh tại: /media/data3/users/luongdth/MulCo-PlantNet/data/processed/PlantDocSplited_depth_AUG/global_image_caption_mapping.json

Loading Training Dataset...
[MultiModalRawDataset] Loaded image-caption mapping: 5256 entries
[MultiModalRawDataset] Total selected images: 2337
[MultiModalRawDataset] Valid samples: 2337
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Matched by external mapping: 2332
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_lea

## 2. Định nghĩa Model và Load Checkpoint cũ

In [26]:
class MulCoEndToEnd(nn.Module):
    def __init__(self, num_classes=28, proj_dim=512):
        super().__init__()
        self.image_backbone = ConvNeXt_CBAM(num_classes=num_classes)
        self.text_backbone = AutoModel.from_pretrained("roberta-base")
        
        
        self.img_proj = nn.Conv2d(1024, proj_dim, kernel_size=1)
        self.txt_proj = nn.Linear(768, proj_dim)
        
        self.fusion_blocks = nn.ModuleList([
            # Sử dụng 1 khối Fusion duy nhất
            MulCoFusionBlock(dim=proj_dim, num_heads=8) for _ in range(1)
        ])
        
        self.classifier = Conv1x1Classifier(in_channels=proj_dim, num_classes=num_classes)

    def forward(self, images, input_ids, attention_mask):
        img_feat = self.image_backbone.forward_features_spatial(images) 
        
        txt_out = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = txt_out.last_hidden_state
        
        img_feat = self.img_proj(img_feat)
        txt_feat = self.txt_proj(txt_feat)
        
        for block in self.fusion_blocks:
            img_feat, txt_feat = block(img_feat, txt_feat, attention_mask)
            
        logits = self.classifier(img_feat)
        
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MulCoEndToEnd(num_classes=28).to(device)

# Đã xóa phần load checkpoint cũ để đảm bảo train từ đầu (Train from scratch)
print("Training from scratch!")

# RoBERTa Tokenizer (hỗ trợ độ dài lên tới 512 tokens)
text_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

Using device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training from scratch!


## 3. Đóng băng các Layer không cần thiết & Cấu hình Optimizer

In [27]:
# Đóng băng lại toàn bộ RoBERTa để tránh học vẹt Text trên tập dữ liệu nhỏ
for param in model.text_backbone.parameters():
    param.requires_grad = False

# MỞ BĂNG SIÊU TINH CHỈNH (Micro Unfreezing) Image Backbone
# Thay vì mở toàn bộ stage 3 khổng lồ gây nhiễu, ta chỉ mở các lớp điều chuẩn (Norm), 
# Attention không gian (CBAM) ở cuối để tập trung vào đốm bệnh mà không làm quên ImageNet.
for name, param in model.image_backbone.named_parameters():
    if "cbam" in name or "norm" in name or "head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

import numpy as np
import torch.nn.functional as F

class CBFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.999, gamma=2.0):
        super().__init__()
        # 1. Tính Số lượng Mẫu Hiệu quả (Effective Number of Samples)
        effective_num = 1.0 - np.power(beta, class_counts)
        weights = (1.0 - beta) / np.array(effective_num)
        
        # Chuẩn hóa weights để tổng của chúng bằng số lượng classes (ổn định gradient)
        weights = weights / np.sum(weights) * len(class_counts)
        
        self.register_buffer('weights', torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        # 2. Áp dụng trọng số Class-Balanced vào Focal Loss
        batch_weights = self.weights[targets]
        cb_focal_loss = batch_weights * focal_loss
        
        return cb_focal_loss.mean()

# Tính toán tần suất xuất hiện của từng lớp từ tập Train
train_labels = [sample["label"] for sample in train_dataset.samples]
class_counts = np.bincount(train_labels, minlength=28)
class_counts = np.maximum(class_counts, 1) # Tránh lỗi chia cho 0
total_samples = len(train_labels)

# Cấu hình hàm Loss và Optimizer
# Hạ beta xuống 0.99 cho tập dataset quy mô nhỏ (vài chục đến vài trăm ảnh/lớp)
criterion = CBFocalLoss(class_counts, beta=0.99, gamma=2.0).to(device)

# Train từ đầu nên có thể dùng Learning Rate lớn hơn so với Fine-tune
LEARNING_RATE = 1e-4 

# Áp dụng Differential Learning Rates (Tốc độ học khác nhau cho từng phần)
optimizer = torch.optim.AdamW([
    # Fusion và Classifier học với LR chuẩn để hội tụ nhanh
    {'params': model.fusion_blocks.parameters(), 'lr': LEARNING_RATE},
    {'params': model.classifier.parameters(), 'lr': LEARNING_RATE},
    {'params': model.img_proj.parameters(), 'lr': LEARNING_RATE},
    {'params': model.txt_proj.parameters(), 'lr': LEARNING_RATE},
    # Khối ConvNeXt mở băng dùng LR cực nhỏ (1/50) để học cực kỳ cẩn thận
    {'params': filter(lambda p: p.requires_grad, model.image_backbone.parameters()), 'lr': LEARNING_RATE / 50.0}
], weight_decay=1e-3)

NUM_EPOCHS = 30
# Thêm Scheduler giảm dần Learning Rate mượt mà qua toàn bộ 30 Epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

best_val_loss = float('inf')
save_dir = os.path.join(PROJECT_ROOT, "archive", "mulco_depth_aug_cb_focal")
os.makedirs(save_dir, exist_ok=True)

# Lưu class mapping để dùng cho validation/inference độc lập sau này
class_mapping_path = os.path.join(save_dir, "class_mapping.json")
with open(class_mapping_path, "w", encoding="utf-8") as f:
    json.dump(train_dataset.idx_to_class, f, indent=4, ensure_ascii=False)
print(f"Đã lưu mapping class -> index tại {class_mapping_path}")

Đã lưu mapping class -> index tại /media/data3/users/luongdth/MulCo-PlantNet/archive/mulco_depth_aug_cb_focal/class_mapping.json


## 4. Bắt đầu Huấn luyện (Fine-tuning Loop)

In [28]:
print("Bắt đầu huấn luyện...")

# Khởi tạo GradScaler cho Automatic Mixed Precision (AMP) giúp giảm một nửa VRAM
scaler = torch.cuda.amp.GradScaler()

ACCUMULATION_STEPS = 4 # Tăng Batch Size ảo lên 32 (8 * 4)

for epoch in range(NUM_EPOCHS):
    # --- TRAIN PHASE ---
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(train_pbar):
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        texts = batch["text"]
        
        # Tokenize text captions with increased max_length
        text_tokens = text_tokenizer(
            texts, padding=True, truncation=True, max_length=256, return_tensors="pt"
        )
        input_ids = text_tokens.input_ids.to(device)
        attn_mask = text_tokens.attention_mask.to(device)
        
        outputs = model(images, input_ids, attn_mask)
        loss = criterion(outputs, labels)
        
        # Scale loss cho Gradient Accumulation
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        # Chỉ cập nhật trọng số sau mỗi ACCUMULATION_STEPS batches
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        train_loss += (loss.item() * ACCUMULATION_STEPS) * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels).item()
        total_train += labels.size(0)
        
        train_pbar.set_postfix({"Loss": f"{loss.item() * ACCUMULATION_STEPS:.4f}"})
        
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")
        for batch in val_pbar:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            texts = batch["text"]
            
            text_tokens = text_tokenizer(
                texts, padding=True, truncation=True, max_length=256, return_tensors="pt"
            )
            input_ids = text_tokens.input_ids.to(device)
            attn_mask = text_tokens.attention_mask.to(device)
            
            outputs = model(images, input_ids, attn_mask)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels).item()
            total_val += labels.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    # Cập nhật Learning Rate
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | LR: {current_lr:.6f} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    # Lưu Model Tốt Nhất
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        save_path = os.path.join(save_dir, "best_fine_tuned_model.pth")
        torch.save(model.state_dict(), save_path)
        print(f"🚀 Model improved! Saved to {save_path}")

print("Hoàn tất huấn luyện Fine-tuning!")

Bắt đầu huấn luyện...


/tmp/ipykernel_337090/4180720046.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/30 [Train]:   0%|          | 0/292 [00:00<?, ?it/s]

Epoch 1/30 [Val]:   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 1/30 | LR: 0.000100 | Train Loss: 0.7460 | Train Acc: 0.4007 | Val Loss: 0.5912 | Val Acc: 0.6847
🚀 Model improved! Saved to /media/data3/users/luongdth/MulCo-PlantNet/archive/mulco_depth_aug_cb_focal/best_fine_tuned_model.pth


...

Epoch 29/30 [Val]:   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 29/30 | LR: 0.000002 | Train Loss: 0.2518 | Train Acc: 0.8283 | Val Loss: 0.1883 | Val Acc: 0.7898
🚀 Model improved! Saved to /media/data3/users/luongdth/MulCo-PlantNet/archive/mulco_depth_aug_cb_focal/best_fine_tuned_model.pth


Epoch 30/30 [Train]:   0%|          | 0/292 [00:00<?, ?it/s]

Epoch 30/30 [Val]:   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 30/30 | LR: 0.000001 | Train Loss: 0.2467 | Train Acc: 0.8442 | Val Loss: 0.2052 | Val Acc: 0.7958
Hoàn tất huấn luyện Fine-tuning!
